# UnitMatchPy with SpikeInterface

This notebook uses SpikeInterface 0.104 or newer. It runs with a small deterministic synthetic dataset by default, so the numerical workflow can be exercised without external data or a GUI.

Install both packages in the same environment with `pip install UnitMatchPy spikeinterface`.

## Configuration

Set `USE_SYNTHETIC_DATA = False` and provide two or more saved Sorting Analyzer paths to use your own data. The analyzers must contain the `random_spikes` and `waveforms` extensions. Unit selection belongs to your analysis pipeline and happens before export.

In [ ]:
from pathlib import Path

USE_SYNTHETIC_DATA = True
ANALYZER_PATHS = [
    Path("path/to/session_1_analyzer"),
    Path("path/to/session_2_analyzer"),
]
EXPORT_DIR = Path("unitmatch_demo_data")
OUTPUT_DIR = Path("unitmatch_demo_results")
RUN_GUI = False
SAVE_RESULTS = False

## Create or load Sorting Analyzers

The synthetic path is deterministic and creates two in-memory analyzers. For real data, load analyzers that already have the required extensions.

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import spikeinterface.full as si

from UnitMatchPy.default_params import get_default_param
from UnitMatchPy.save_utils import make_UnitMatch_folder_from_sorting_analyzers
from UnitMatchPy.utils import get_probe_geometry


def make_synthetic_analyzer(seed):
    recording, sorting = si.generate_ground_truth_recording(
        durations=[5.0],
        sampling_frequency=30_000.0,
        num_channels=16,
        num_units=8,
        seed=seed,
    )
    analyzer = si.create_sorting_analyzer(
        sorting=sorting, recording=recording, format="memory", sparse=True
    )
    analyzer.compute(
        "random_spikes", method="uniform", max_spikes_per_unit=200, seed=seed
    )
    analyzer.compute("waveforms", ms_before=1.0, ms_after=1.0)
    return analyzer


if USE_SYNTHETIC_DATA:
    source_analyzers = [make_synthetic_analyzer(410), make_synthetic_analyzer(411)]
else:
    source_analyzers = [si.load_sorting_analyzer(path) for path in ANALYZER_PATHS]

# Replace these arrays with unit IDs selected by your own curation criteria.
selected_unit_ids = [analyzer.unit_ids for analyzer in source_analyzers]
analyzers = [
    analyzer.select_units(unit_ids)
    for analyzer, unit_ids in zip(source_analyzers, selected_unit_ids)
]
unit_ids_per_session = [analyzer.unit_ids for analyzer in analyzers]
spike_times_per_unit = [
    analyzer.sorting.get_unit_spike_train(unit_id=unit_id)
    / analyzer.sampling_frequency
    for analyzer in analyzers
    for unit_id in analyzer.unit_ids
]

make_UnitMatch_folder_from_sorting_analyzers(
    analyzers=analyzers, save_dir=EXPORT_DIR, overwrite=True
)

## Load exported waveforms and run UnitMatch

In [ ]:
import UnitMatchPy.assign_unique_id as aid
import UnitMatchPy.bayes_functions as bf
import UnitMatchPy.overlord as ov
import UnitMatchPy.save_utils as su
import UnitMatchPy.utils as util

wave_paths = [EXPORT_DIR / f"Session{index}" for index in range(len(analyzers))]
channel_pos = [np.load(path / "channel_locations.npy") for path in wave_paths]

with (wave_paths[0] / "waveform_params.json").open(encoding="utf-8") as stream:
    waveform_params = json.load(stream)

param = get_default_param()
param.update(waveform_params)
param["waveidx"] = np.asarray(param["waveidx"], dtype=int)
param = get_probe_geometry(channel_pos[0], param)

waveform, session_id, session_switch, within_session, unit_ids, param = (
    util.load_waveforms(wave_paths, unit_ids_per_session, param)
)
clus_info = {
    "good_units": unit_ids,
    "session_switch": session_switch,
    "session_id": session_id,
    "original_ids": np.concatenate(unit_ids),
    "spike_times": spike_times_per_unit,
}

extracted_wave_properties = ov.extract_parameters(
    waveform, channel_pos, clus_info, param
)
total_score, candidate_pairs, scores_to_include, predictors = ov.extract_metric_scores(
    extracted_wave_properties, session_switch, within_session, param, niter=2
)

prior_match = 1 - param["n_expected_matches"] / param["n_units"] ** 2
priors = np.array((prior_match, 1 - prior_match))
labels = candidate_pairs.astype(int)
conditions = np.unique(labels)
parameter_kernels = bf.get_parameter_kernels(
    scores_to_include, labels, conditions, param, add_one=1
)
probability = bf.apply_naive_bayes(
    parameter_kernels, priors, predictors, param, conditions
)
output_prob_matrix = probability[:, 1].reshape(param["n_units"], param["n_units"])

In [ ]:
match_threshold = 0.75
param["match_threshold"] = match_threshold
util.evaluate_output(
    output_prob_matrix,
    param,
    within_session,
    session_switch,
    match_threshold=match_threshold,
)

output_threshold = (output_prob_matrix > match_threshold).astype(int)
plt.imshow(output_threshold, cmap="Greys")
plt.title(f"Matches at probability > {match_threshold}")
plt.show()

## Optional manual GUI curation

The GUI requires an interactive desktop. Leave `RUN_GUI = False` for headless execution; automatic matches will be used instead.

In [ ]:
matches = np.argwhere(output_threshold == 1)
matches_curated = None

if RUN_GUI:
    import UnitMatchPy.GUI as gui

    gui.process_info_for_GUI(
        output_prob_matrix,
        match_threshold,
        scores_to_include,
        total_score,
        extracted_wave_properties["amplitude"],
        extracted_wave_properties["spatial_decay"],
        extracted_wave_properties["avg_centroid"],
        extracted_wave_properties["avg_waveform"],
        extracted_wave_properties["avg_waveform_per_tp"],
        extracted_wave_properties["good_wave_idxs"],
        extracted_wave_properties["max_site"],
        extracted_wave_properties["max_site_mean"],
        waveform,
        within_session,
        channel_pos,
        clus_info,
        param,
    )
    is_match, not_match, matches_gui = gui.run_GUI()
    matches_curated = util.curate_matches(
        matches_gui, is_match, not_match, mode="and"
    )

## Optional result export

In [ ]:
unique_ids = aid.assign_unique_id(output_prob_matrix, param, clus_info)

if SAVE_RESULTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    su.save_to_output(
        OUTPUT_DIR,
        scores_to_include,
        matches_curated if matches_curated is not None else matches,
        output_prob_matrix,
        extracted_wave_properties["avg_centroid"],
        extracted_wave_properties["avg_waveform"],
        extracted_wave_properties["avg_waveform_per_tp"],
        extracted_wave_properties["max_site"],
        total_score,
        output_threshold,
        clus_info,
        param,
        UIDs=unique_ids,
        matches_curated=matches_curated,
        save_match_table=True,
    )